# Projet 1 — LightGCN : Systèmes de Recommandation
**ANI-IA 4 | BILOA ABADJECK Paolo | ENSPY Yaoundé**

---

Ce notebook couvre l'intégralité du projet :
1. Chargement et exploration du dataset MovieLens
2. Construction du graphe biparti
3. Implémentation et entraînement de LightGCN
4. Évaluation (Recall@K, NDCG@K)
5. Visualisation t-SNE des embeddings
6. Recommandations Top-10 personnalisées

In [ ]:
# ── Installation des dépendances (si nécessaire) ──
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch', 'numpy', 'pandas', 'scipy',
                'scikit-learn', 'matplotlib', 'seaborn', 'tqdm'])

In [ ]:
# ── Imports ──
import os, sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
print(f'PyTorch : {torch.__version__}')

## 1. Chargement et Exploration des Données

In [ ]:
from data_loader import load_ratings, preprocess_ratings, train_test_split_ratings

# Chargement
df_raw = load_ratings('../data')
print(df_raw.head())
print(f'\nShape : {df_raw.shape}')
print(f'Colonnes : {df_raw.columns.tolist()}')

In [ ]:
# Distribution des ratings
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df_raw['rating'].value_counts().sort_index().plot(kind='bar', ax=axes[0],
    color='#2196F3', edgecolor='white')
axes[0].set_title('Distribution des Ratings', fontsize=13)
axes[0].set_xlabel('Note')
axes[0].set_ylabel('Fréquence')

user_counts = df_raw.groupby('userId')['movieId'].count()
axes[1].hist(user_counts, bins=50, color='#4CAF50', edgecolor='white')
axes[1].set_title('Films notés par utilisateur', fontsize=13)
axes[1].set_xlabel('Nombre de films')
axes[1].set_ylabel('Fréquence')

plt.tight_layout()
plt.show()

In [ ]:
# Prétraitement
df = preprocess_ratings(df_raw, min_interactions=5)
df_train, df_test = train_test_split_ratings(df, test_ratio=0.2)

n_users = df.attrs['n_users']
n_items = df.attrs['n_items']

print(f'\nUtilisateurs : {n_users}')
print(f'Items        : {n_items}')
print(f'Train        : {len(df_train):,}')
print(f'Test         : {len(df_test):,}')
print(f'Densité      : {len(df_train) / (n_users * n_items):.6f}')

## 2. Construction du Graphe Biparti

On construit la matrice d'adjacence :
$$\tilde{A} = \begin{pmatrix} 0 & R \\ R^\top & 0 \end{pmatrix}$$

Puis la normalisation symétrique :
$$\hat{A} = D^{-1/2} \tilde{A} D^{-1/2}$$

In [ ]:
from graph_builder import build_graph

A_hat = build_graph(df_train, n_users, n_items, device=device)
print(f'\nMatrice A_hat : {list(A_hat.shape)}')
print(f'Taille       : ({n_users} users + {n_items} items) x ({n_users} + {n_items})')

## 3. Modèle LightGCN

Propagation multicouches :
$$E^{(k)} = \hat{A} \cdot E^{(k-1)}$$

Agrégation finale :
$$E^* = \frac{1}{K+1} \sum_{k=0}^{K} E^{(k)}$$

In [ ]:
from model import LightGCN

model = LightGCN(n_users=n_users, n_items=n_items, embed_dim=64, n_layers=3)
total_params = sum(p.numel() for p in model.parameters())

print(f'Architecture LightGCN')
print(f'  embed_dim  : 64')
print(f'  n_layers   : 3')
print(f'  Paramètres : {total_params:,}')
print(f'\n{model}')

## 4. Entraînement avec BPR Loss

$$\mathcal{L}_{BPR} = -\sum_{(u,i,j)} \ln \sigma(\hat{y}_{ui} - \hat{y}_{uj}) + \lambda \|E^{(0)}\|^2$$

In [ ]:
from trainer import Trainer

trainer = Trainer(
    model      = model,
    A_hat      = A_hat,
    df_train   = df_train,
    n_items    = n_items,
    lr         = 1e-3,
    reg_lambda = 1e-4,
    batch_size = 1024,
    device     = device,
    save_dir   = '../outputs'
)

history = trainer.fit(n_epochs=100, verbose=10)

In [ ]:
# Courbe de perte
plt.figure(figsize=(10, 4))
plt.plot(history['epoch'], history['loss'], color='#2196F3', linewidth=2)
plt.fill_between(history['epoch'], history['loss'], alpha=0.1, color='#2196F3')
plt.xlabel('Époque', fontsize=12)
plt.ylabel('Perte BPR', fontsize=12)
plt.title('Courbe de Convergence — LightGCN', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/loss_curve.png', dpi=150)
plt.show()

## 5. Évaluation — Recall@K et NDCG@K

In [ ]:
from evaluator import evaluate
from data_loader import get_user_item_sets, get_test_ground_truth

# Recharger le meilleur modèle
model.load_state_dict(torch.load('../outputs/lightgcn_best.pt', map_location=device))
model = model.to(device)

user_items   = get_user_item_sets(df_train)
ground_truth = get_test_ground_truth(df_test)

results = evaluate(
    model=model, A_hat=A_hat,
    ground_truth=ground_truth,
    train_user_items=user_items,
    k=10, device=device
)

print('\n=== Résultats Finaux ===')
for k, v in results.items():
    print(f'  {k} : {v:.4f}')

In [ ]:
# Diagramme des métriques
fig, ax = plt.subplots(figsize=(7, 4))
keys = list(results.keys())
vals = list(results.values())
colors = ['#2196F3', '#4CAF50']
bars = ax.bar(keys, vals, color=colors, edgecolor='white', width=0.4)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{val:.4f}', ha='center', fontsize=12, fontweight='bold')
ax.set_ylim(0, max(vals)*1.4)
ax.set_ylabel('Score')
ax.set_title('Résultats — LightGCN sur MovieLens', fontsize=13)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/metrics_bar.png', dpi=150)
plt.show()

## 6. Recommandations Top-10 personnalisées

In [ ]:
from evaluator import show_recommendations

# Charger le mapping idx -> titre film
import os
movies_path = '../data/ml-latest-small/movies.csv'
idx2movie = None

if os.path.exists(movies_path):
    movies_df = pd.read_csv(movies_path)
    item2idx  = df.attrs.get('item2idx', {})
    idx2movie = {v: movies_df.loc[movies_df['movieId']==k, 'title'].values[0]
                 for k, v in item2idx.items()
                 if k in movies_df['movieId'].values}

# Recommandations pour 5 utilisateurs
sample_users = list(ground_truth.keys())[:5]
for u in sample_users:
    show_recommendations(model, A_hat, u, user_items, idx2movie, k=10)

## 7. Visualisation t-SNE des Embeddings

In [ ]:
from visualizer import plot_tsne_embeddings

plot_tsne_embeddings(
    model, A_hat,
    n_users=n_users, n_items=n_items,
    n_sample_users=150, n_sample_items=250,
    save=True
)

from IPython.display import Image
Image('../outputs/tsne_embeddings.png')

## 8. Résumé et Conclusion

In [ ]:
print('=' * 50)
print('  RÉSUMÉ DU PROJET 1 — LightGCN')
print('=' * 50)
print(f'  Dataset       : MovieLens ml-latest-small')
print(f'  Utilisateurs  : {n_users}')
print(f'  Items (films) : {n_items}')
print(f'  Interactions  : {len(df_train):,} (train)')
print(f'  embed_dim     : 64 | n_layers : 3')
print(f'  Époques       : 100')
print()
for k, v in results.items():
    print(f'  {k:15s} : {v:.4f}')
print('=' * 50)